# Day 26 Capstone: Nimbus Customer-Feedback Intelligence

**Offline, CPU-first pipeline** | TF-IDF + logistic regression, MiniLM semantic search, FLAN-T5-small summarization and topic labeling. All reviews are embedded in this notebook. **No API calls or dataset URLs.**

**Environment note:** The pretrained models must already exist in the local Hugging Face cache. This notebook explicitly enables offline mode and will raise an actionable error if either model is missing. The first run cannot download models while remaining fully offline. Run all cells in order in a lab image with the cached weights.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
import warnings, re, textwrap
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
RNG = 42
np.random.seed(RNG)
REVIEWS = [
 ('The delivery arrived two days early and the packaging was flawless.', 'positive'),
 ('Battery life is incredible, easily lasts me a full working day.', 'positive'),
 ('Setup took five minutes and the app walked me through every step.', 'positive'),
 ('Customer support answered in under a minute and fixed my issue.', 'positive'),
 ('Build quality feels premium, the aluminium body is gorgeous.', 'positive'),
 ('Best purchase this year, the sound quality genuinely surprised me.', 'positive'),
 ('Screen is bright and crisp even in direct sunlight, love it.', 'positive'),
 ('Refund was processed instantly, no questions asked. Great service.', 'positive'),
 ('The subscription is worth every penny, new features every month.', 'positive'),
 ('Lightweight and comfortable, I forget I am even wearing it.', 'positive'),
 ('Pairing with my laptop was seamless and it just works.', 'positive'),
 ('Fantastic value for the price, would recommend to friends.', 'positive'),
 ('The firmware update fixed the lag and now it is buttery smooth.', 'positive'),
 ('Packaging was recyclable and the quick-start guide was clear.', 'positive'),
 ('Warranty claim was handled in a single email, very impressed.', 'positive'),
 ('Camera photos are sharp and colours look natural, thrilled.', 'positive'),
 ('Delivery was five days late and no one told me why.', 'negative'),
 ('The battery drains overnight even when the device is switched off.', 'negative'),
 ('App keeps crashing every time I open the settings menu.', 'negative'),
 ('Support put me on hold for forty minutes then dropped the call.', 'negative'),
 ('The plastic casing cracked within a week of normal use.', 'negative'),
 ('Overpriced for what it is, I expected far better at this cost.', 'negative'),
 ('Screen has a dead pixel right in the middle, very distracting.', 'negative'),
 ('They refused my refund and blamed me for the defect.', 'negative'),
 ('The subscription auto-renewed without any warning email.', 'negative'),
 ('It is heavy and uncomfortable after an hour of wearing it.', 'negative'),
 ('Bluetooth disconnects randomly every few minutes, infuriating.', 'negative'),
 ('Terrible value, it broke twice and shipping cost me extra.', 'negative'),
 ('The latest firmware update bricked my device completely.', 'negative'),
 ('Packaging was crushed and the manual was missing entirely.', 'negative'),
 ("Warranty was denied because the form was 'filled in wrong'.", 'negative'),
 ('Camera is grainy in low light and the autofocus hunts forever.', 'negative'),
 ('Shipping was quick but the box arrived dented on one corner.', 'negative'),
 ('Good sound but the app notifications are relentless and annoying.', 'negative'),
 ('Love the design though the battery could honestly be better.', 'positive'),
 ('Works well overall, only wish support replied a little faster.', 'positive'),
 ('Cheap feeling buttons let down an otherwise solid product.', 'negative'),
 ('The return shipping label never arrived so I am stuck with it.', 'negative'),
 ('Crystal clear calls and the mic cancels background noise well.', 'positive'),
 ('Constant software bugs make an expensive product feel like a beta.', 'negative'),
]
df = pd.DataFrame(REVIEWS, columns=['text','sentiment'])
print('Shape:', df.shape)
print(df['sentiment'].value_counts().to_string())
display(df.head(3))

## 1. Normalize text while preserving the original review

In [ ]:
STOP = {'the','a','an','and','or','but','is','are','was','were','it','its','i','me','my','of','to','in','for','on','at','this','that','with','as','so','even','then','far','than','after','every','no','not','up','out','you'}
def normalize(text):
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    return [token for token in text.split() if token and token not in STOP]
df['tokens'] = df['text'].apply(normalize)
df['clean'] = df['tokens'].apply(' '.join)
print('RAW:', df.loc[0,'text'])
print('TOKENS:', df.loc[0,'tokens'])
print('Average tokens:', round(df['tokens'].str.len().mean(),1))

## 2. Classical baseline: TF-IDF and logistic regression

Ground-truth sentiment is used for training and dashboard counts. The test split is held out and vectorization is fitted only on training data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

y = (df['sentiment'] == 'positive').astype(int)
Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(df['clean'], y, df.index, test_size=.30, stratify=y, random_state=RNG)
vec = TfidfVectorizer(ngram_range=(1,2), min_df=1)
Xtr_v = vec.fit_transform(Xtr)
Xte_v = vec.transform(Xte)
clf = LogisticRegression(C=1.0, max_iter=2000, random_state=RNG).fit(Xtr_v, ytr)
pred = clf.predict(Xte_v)
print(f'Test accuracy: {accuracy_score(yte,pred):.3f} (n={len(yte)})')
print(classification_report(yte,pred, target_names=['negative','positive'], zero_division=0))
vocab = np.array(vec.get_feature_names_out())
coefs = clf.coef_[0]
print('Positive features:', vocab[np.argsort(coefs)[-6:][::-1]].tolist())
print('Negative features:', vocab[np.argsort(coefs)[:6]].tolist())
errors = df.loc[idx_te[np.asarray(pred) != np.asarray(yte)], ['text','sentiment']]
print('Held-out misclassifications:')
display(errors)

## 3. Semantic embeddings and cosine similarity

The embedding model is loaded **only from local cache**. No downloads are permitted.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
try:
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu', local_files_only=True)
except Exception as exc:
    raise RuntimeError('MiniLM weights are not available in the local model cache. Ask the lab administrator to pre-cache sentence-transformers/all-MiniLM-L6-v2.') from exc
emb = embedder.encode(df['text'].tolist(), normalize_embeddings=True, show_progress_bar=False)
assert emb.shape == (40,384)
print('Embedding matrix:', emb.shape)
def search(query, k=3, exclude_index=None):
    q = embedder.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q,emb)[0]
    if exclude_index is not None:
        sims[exclude_index] = -np.inf
    order = np.argsort(sims)[::-1][:max(0,min(k,len(df)-(exclude_index is not None)))]
    return [(round(float(sims[i]),3), df.loc[i,'text']) for i in order]
for query in ['problems with the battery','the shipping was slow']:
    print('\nQuery:',query)
    for score, review in search(query): print(score, review)

## 4. Abstractive summarization with FLAN-T5-small

Small models have limited context. Summarize short groups first, then synthesize the group summaries instead of silently truncating the 22 negative reviews.

In [ ]:
from transformers import pipeline
try:
    summarizer = pipeline('text2text-generation', model='google/flan-t5-small', tokenizer='google/flan-t5-small', device=-1, local_files_only=True)
except Exception as exc:
    raise RuntimeError('FLAN-T5-small is not available locally. Ask the lab administrator to pre-cache google/flan-t5-small.') from exc
neg_reviews = df.loc[df.sentiment=='negative','text'].tolist()
def generate(prompt, max_new_tokens=80):
    return summarizer(prompt, max_new_tokens=max_new_tokens, do_sample=False, truncation=True)[0]['generated_text'].strip()
partial_summaries = []
for start in range(0,len(neg_reviews),4):
    batch_text = ' '.join(neg_reviews[start:start+4])
    partial_summaries.append(generate('Summarize the customer complaints briefly: '+batch_text, 55))
summary = generate('Summarize the main customer problems in a short paragraph: '+' '.join(partial_summaries), 100)
print('COMPLAINT SUMMARY:')
print(textwrap.fill(summary,88))
print('Note: check this generated summary against the actual reviews before acting.')

## 5. Zero-shot and few-shot topic classification

The allowed set includes `other` because reviews also discuss cameras, screens, warranty, and Bluetooth. Unknown model outputs are marked `other` rather than silently assigned to an unrelated topic.

In [ ]:
TOPICS = ['shipping','battery','app/software','support','build quality','price','other']
def snap_topic(answer):
    answer = answer.strip().lower().strip(' .,:;"\'')
    if answer in TOPICS: return answer
    aliases = {'app':'app/software','software':'app/software','delivery':'shipping','customer service':'support','quality':'build quality','cost':'price'}
    return aliases.get(answer,'other')
def label_topic(review, few_shot=False):
    examples = ('Review: Delivery was late. Topic: shipping\nReview: Battery drains too fast. Topic: battery\n' if few_shot else '')
    prompt = ('Choose exactly one topic: '+', '.join(TOPICS)+'.\n'+examples+'Review: '+review+'\nTopic:')
    return snap_topic(generate(prompt, 12))
df['topic'] = df['text'].apply(label_topic)
display(df[['text','sentiment','topic']].head(8))
print(df['topic'].value_counts().to_string())
print('Few-shot example:', label_topic('My parcel arrived late.',few_shot=True))

## 6. Dashboard in words

Counts use the supplied ground-truth sentiment labels, not model-predicted sentiment. Topic labels are generated and should be audited.

In [ ]:
n=len(df)
pos=int((df.sentiment=='positive').sum())
neg=n-pos
print('='*60)
print('NIMBUS - CUSTOMER FEEDBACK DASHBOARD (OFFLINE)')
print('='*60)
print(f'Reviews analysed: {n}')
print(f'Sentiment split: {pos} positive ({pos/n:.0%}) | {neg} negative ({neg/n:.0%})')
neg_by_topic=df[df.sentiment=='negative'].groupby('topic').size().sort_values(ascending=False)
print('\nTop complaint areas (model-assigned topics):')
for topic,cnt in neg_by_topic.items():
    print(f'{topic:14s} {cnt:2d} {"#"*int(cnt)}')
if len(neg_by_topic):
    worst=neg_by_topic.index[0]
    example=df[(df.sentiment=='negative') & (df.topic==worst)].iloc[0]['text']
    print(f'\nLoudest labeled problem: {worst}. Example: {example!r}')
print('Caution: 40 fictional reviews, noisy automated topic labels; ties and sample bias limit conclusions.')

## 7. Compare the three families on a mixed review

In [ ]:
sample='Love the design though the battery could honestly be better.'
print('SAME REVIEW, THREE LENSES\n'+'-'*60)
print('Review:',sample)
p=float(clf.predict_proba(vec.transform([' '.join(normalize(sample))]))[0,1])
print(f'1. CLASSICAL: P(positive) = {p:.2f} (uncalibrated on tiny dataset)')
print('2. EMBEDDING: nearest different review:')
for score,txt in search(sample,k=1,exclude_index=34):print(score,txt)
print('3. LLM topic:',label_topic(sample))
print('\nReflection questions: TF-IDF strengths/limits? Embedding value? LLM value? Mixed sentiment? Compute trade-offs?')

## Written comparison: classical NLP vs embeddings vs small LLM

**A. Where TF-IDF is enough.** TF-IDF plus logistic regression is a sensible, fast, explainable first-pass sentiment baseline. Its learned weights show which surface expressions drive the prediction. It is not a robust semantic model: negation, mixed opinions, and paraphrases can be missed, especially with only 28 training reviews.

**B. What embeddings add.** MiniLM maps different wordings of a similar issue into nearby vectors. A query such as “problems with the battery” can retrieve complaints phrased as “drains overnight” or “could be better.” This enables review discovery without training a separate classifier for every query. Cosine similarity is not proof that two reviews make the same claim.

**C. What the LLM adds.** FLAN-T5-small can synthesize complaint themes into a readable paragraph and assign labels using instructions instead of labeled topic examples. The output is flexible but can hallucinate, omit details, or fail to follow the allowed label set.

**D. Mixed sentiment.** “Love the design though the battery could honestly be better” contains both praise and a product weakness. A single binary sentiment score hides that nuance; embedding neighbors can surface battery-related feedback, while a topic label can focus attention on the battery. None of these three outputs alone is a reliable aspect-level sentiment analysis. An aspect-based model or two-label annotation would improve this.

**E. Cost and deployment.** At inference, TF-IDF/logistic regression is usually cheapest, MiniLM embedding inference is more expensive, and repeated FLAN-T5 generation is generally slowest on CPU. For 10,000 weekly reviews, I would deploy the classical baseline first, embed only when semantic search adds operational value, and summarize clustered complaints periodically rather than running a generative model on every review.

## Final reflection (approximately 240 words)

For a team receiving 10,000 customer reviews each week on a tight compute budget, I would retain lightweight normalization, TF-IDF, and logistic regression as a fast sentiment baseline. These components are inexpensive, easy to audit, and suitable for batch processing. I would keep MiniLM embeddings for targeted semantic search, deduplication, and discovering recurring issues that use different wording. I would avoid generating an LLM summary for every individual review. Instead, I would group reviews by topic and generate periodic summaries from representative samples, with a human checking the output. This reduces CPU time and limits unsupported claims.

The review “Love the design though the battery could honestly be better” demonstrates a weakness. Its ground-truth label is positive, but it contains a clear battery complaint. A single sentiment label or topic cannot represent both aspects accurately. Better preprocessing alone would not solve that problem because the missing information is structural: one sentence expresses two opinions. I would first add aspect-level tags for design and battery, then evaluate a more capable instruction-tuned model only if the additional accuracy justifies slower inference and higher memory use. I would inspect actual held-out errors before claiming a model improvement.

If the dashboard identifies battery complaints as the largest category, a product manager could commission battery-drain diagnostics and review recent firmware changes. If another topic leads, the same decision process should target that verified category instead. The main risk is overreacting to 40 fictional reviews: the sample is small, nonrepresentative, and the topic labels can be wrong. Before committing resources, I would validate labels, examine a larger representative sample, and track complaint trends over time.